# MH-02. Y (candidate-node labels) + R (observed-mask)

Row-aligned to `spine.parquet` (same person_id set, order, count; ascending by person_id).

Rules:
- Positive = rule-of-two: node recorded on >= 2 distinct calendar days -> Y = 1; else Y = 0.
- Index-category drop: every node whose 3-digit parent == patient's index_signal_type -> Y = NaN.
- 306.1 -> Y = NaN for males.
- R = ~isnan(Y); assert count(R != ~isnan(Y)) == 0.
- all_negative / all_masked spine flags filled here.


## 1. Load spine, node table, dx events

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path('output/mh')
spine = pd.read_parquet(OUTPUT_DIR / 'spine.parquet')
node_table = pd.read_parquet(OUTPUT_DIR / 'mh_node_table.parquet')
mh_dx = pd.read_parquet(OUTPUT_DIR / 'mh_dx_events.parquet')

SEX_MALE_VALUES = {'M','Male','MALE','male'}   # confirm against person.gender_source_value
NODE_306_1 = '306.1'

nodes = sorted(node_table['phecode'].tolist(), key=lambda s: float(s))
persons = spine['person_id'].tolist()
print('cohort persons:', len(persons), '| nodes:', len(nodes))

# Restrict dx to cohort persons only.
mh_dx = mh_dx[mh_dx['person_id'].isin(set(persons))].copy()
mh_dx['dx_day'] = pd.to_datetime(mh_dx['dx_date']).dt.normalize()


## 2. Rule-of-two positives

In [ ]:
# distinct calendar days per (person, node); >= 2 -> positive.
distinct_days = (mh_dx.groupby(['person_id','phecode'])['dx_day']
                 .nunique().rename('n_days').reset_index())
pos = distinct_days[distinct_days['n_days'] >= 2][['person_id','phecode']]
pos['pos'] = 1

# Dense person x node frame, default 0 (eligible negative).
Y = pd.DataFrame(0, index=pd.Index(persons, name='person_id'), columns=nodes, dtype='float64')
pos_wide = pos.pivot_table(index='person_id', columns='phecode', values='pos', fill_value=0)
pos_wide = pos_wide.reindex(index=persons, columns=nodes, fill_value=0)
Y = Y.add(pos_wide, fill_value=0).clip(upper=1)
print('Y positives total:', int(np.nansum(Y.values)))


## 3. Masking: index-category drop + 306.1 males

In [ ]:
parent_of = dict(zip(node_table['phecode'], node_table['parent3']))
idx_sig = spine.set_index('person_id')['index_signal_type']
sex = spine.set_index('person_id')['sex']

# Index-category drop: for each patient, all nodes whose parent3 == index_signal_type -> NaN.
node_parents = np.array([parent_of[n] for n in nodes])
Yv = Y.values
pid_index_sig = idx_sig.reindex(persons).values
for j, p3 in enumerate(node_parents):
    Yv[pid_index_sig == p3, j] = np.nan

# 306.1 -> NaN for males.
if NODE_306_1 in nodes:
    j = nodes.index(NODE_306_1)
    is_male = sex.reindex(persons).isin(SEX_MALE_VALUES).values
    Yv[is_male, j] = np.nan

Y = pd.DataFrame(Yv, index=Y.index, columns=nodes)
print('masked (NaN) cells:', int(np.isnan(Y.values).sum()))


## 4. R = ~isnan(Y) with zero-mismatch assertion

In [ ]:
R = (~np.isnan(Y.values)).astype('int8')
R = pd.DataFrame(R, index=Y.index, columns=nodes)

mismatch = int((R.values != (~np.isnan(Y.values)).astype('int8')).sum())
print('R != ~isnan(Y) mismatch count:', mismatch)
assert mismatch == 0, 'R must equal ~isnan(Y)'
assert not np.isnan(R.values).any(), 'R must never be NaN'


## 5. Fill spine all_negative / all_masked, save row-aligned

In [ ]:
scored = ~np.isnan(Y.values)
all_masked = (~scored).all(axis=1)
# all_negative: every scored node == 0 (and at least one scored).
row_pos = np.nansum(np.where(scored, Y.values, 0), axis=1)
all_negative = scored.any(axis=1) & (row_pos == 0)

spine = spine.set_index('person_id')
spine.loc[persons, 'all_masked'] = all_masked
spine.loc[persons, 'all_negative'] = all_negative
spine = spine.reset_index()

# Emit ascending by person_id, identical row set/order across spine/Y/R.
order = spine.sort_values('person_id')['person_id'].tolist()
assert order == sorted(persons)
Y = Y.reindex(order); R = R.reindex(order)
Y_out = Y.reset_index(); R_out = R.reset_index()

spine.sort_values('person_id').to_parquet(OUTPUT_DIR / 'spine.parquet', index=False)
Y_out.to_parquet(OUTPUT_DIR / 'Y_nodes.parquet', index=False)
R_out.to_parquet(OUTPUT_DIR / 'R_mask.parquet', index=False)
print('Y:', Y_out.shape, '| R:', R_out.shape)
print('all_masked:', int(all_masked.sum()), '| all_negative:', int(all_negative.sum()))
